In [ ]:
# Install required packages
!pip install langchain langchain-openai langsmith openai -q

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"]    = userdata.get("OPENAI_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except Exception:
    os.environ["OPENAI_API_KEY"]    = "sk-..."         
    os.environ["LANGSMITH_API_KEY"] = "lsv2_pt_..."     

os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_PROJECT"]     = "AI-Resume-Screening"   
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"

print("✅ Environment configured.")
print(f"   Tracing enabled  : {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"   LangSmith project: {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
JOB_DESCRIPTION = """
Job Title: Data Scientist
Company  : TechCorp Analytics

Required Skills:
- Python (NumPy, Pandas, Scikit-learn)
- Machine Learning (supervised & unsupervised)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data wrangling
- Data visualization (Matplotlib, Seaborn, or Tableau)
- MLflow or similar experiment tracking
- Strong statistics and probability knowledge

Experience Required: 2+ years in a data science or ML role
Education: B.Tech / M.Tech / MSc in CS, Statistics, or related field
"""

RESUME_STRONG = """
Name: Aisha Patel
Education: M.Tech in Computer Science, IIT Bombay (2022)

Experience:
- Data Scientist at DataWave (3 years): Built ML pipelines using Scikit-learn,
  XGBoost; reduced churn by 18%. Managed SQL databases.
- Research Intern at Google (6 months): NLP model fine-tuning with PyTorch.

Skills:
- Python (Pandas, NumPy, Scikit-learn, PyTorch, TensorFlow)
- SQL, Spark, MLflow, Docker
- Tableau, Matplotlib, Seaborn
- Statistics, A/B Testing, Hypothesis Testing

Projects:
- Customer Churn Prediction (XGBoost + SHAP explainability)
- Image Classifier CNN (PyTorch, 94% accuracy)
"""

RESUME_AVERAGE = """
Name: Rahul Sharma
Education: B.Tech in Information Technology, VIT (2023)

Experience:
- Data Analyst Intern at FinTech Ltd (6 months): Created dashboards in Tableau,
  wrote SQL queries, basic Python scripting.

Skills:
- Python (Pandas, Matplotlib)
- SQL (intermediate)
- Tableau
- Basic Scikit-learn (Linear Regression, KNN)

Projects:
- Sales Forecasting with Linear Regression
- EDA on Titanic dataset
"""

RESUME_WEAK = """
Name: Priya Verma
Education: B.Com, Delhi University (2021)

Experience:
- Sales Executive at RetailMart (2 years): Customer handling, inventory management.
- Freelance data entry (1 year): MS Excel, basic data sorting.

Skills:
- MS Excel, PowerPoint
- Basic internet research
- Tally ERP

Projects:
- Excel-based sales tracker
"""

candidates = [
    {"name": "Aisha Patel",  "type": "Strong",  "resume": RESUME_STRONG},
    {"name": "Rahul Sharma", "type": "Average", "resume": RESUME_AVERAGE},
    {"name": "Priya Verma",  "type": "Weak",    "resume": RESUME_WEAK},
]

print(f"✅ Loaded {len(candidates)} resumes + 1 Job Description.")

In [ ]:
from langchain.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert HR analyst. Extract information ONLY from the resume provided.
Do NOT infer or assume skills that are not explicitly mentioned.

RESUME:
{resume}

Extract and return a JSON object with these fields:
{{
  "name": "candidate name",
  "education": "highest degree and field",
  "years_experience": <number or 0 if fresher>,
  "skills": ["list", "of", "technical", "skills"],
  "tools": ["tools", "and", "frameworks"],
  "domains": ["domain", "areas"]
}}

Return ONLY valid JSON. No explanation. No markdown fences.
"""
)

matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a strict job-fit evaluator.

CANDIDATE PROFILE (JSON):
{extracted_profile}

JOB DESCRIPTION:
{job_description}

Compare the candidate profile against the job requirements.
Return a JSON object:
{{
  "matched_skills": ["skills present in both profile and JD"],
  "missing_skills": ["skills required by JD but absent in profile"],
  "extra_skills": ["skills in profile but not required by JD"],
  "experience_match": true/false,
  "education_match": true/false
}}

Return ONLY valid JSON. No explanation. No markdown fences.
"""
)

scoring_prompt = PromptTemplate(
    input_variables=["match_analysis", "job_description"],
    template="""
You are a precise scoring engine for resume evaluation.

MATCH ANALYSIS (JSON):
{match_analysis}

JOB DESCRIPTION:
{job_description}

Assign a fit score from 0 to 100 using this rubric:
- Skills match   : 40 points
- Experience     : 30 points
- Education      : 20 points
- Extra/bonus skills: 10 points

Return a JSON object:
{{
  "score": <integer 0–100>,
  "skills_points":     <0–40>,
  "experience_points": <0–30>,
  "education_points":  <0–20>,
  "bonus_points":      <0–10>
}}

Return ONLY valid JSON. No explanation. No markdown fences.
"""
)

explanation_prompt = PromptTemplate(
    input_variables=["extracted_profile", "match_analysis", "score_breakdown", "job_description"],
    template="""
You are an HR consultant writing a clear, factual candidate evaluation report.
Base your explanation ONLY on the provided data. Do NOT invent skills or facts.

--- FEW-SHOT EXAMPLE ---
Score: 85 | Explanation: The candidate strongly matches the role. They have 4/5 required
skills, 3 years relevant experience (meets 2+ requirement), and a relevant M.Tech degree.
Missing skill: MLflow. Bonus: Docker knowledge adds value.
Recommendation: SHORTLIST
--- END EXAMPLE ---

Now evaluate this candidate:

EXTRACTED PROFILE:
{extracted_profile}

MATCH ANALYSIS:
{match_analysis}

SCORE BREAKDOWN:
{score_breakdown}

JOB DESCRIPTION:
{job_description}

Return a JSON object:
{{
  "total_score": <integer from score breakdown>,
  "strengths": "2-3 sentence summary of candidate strengths",
  "gaps": "1-2 sentence summary of missing areas",
  "explanation": "3-4 sentence overall evaluation",
  "recommendation": "SHORTLIST" | "MAYBE" | "REJECT"
}}

Return ONLY valid JSON. No markdown fences.
"""
)

print("✅ All 4 PromptTemplates created (extraction, matching, scoring, explanation).")

In [ ]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-3.5-turbo",   
    temperature=0,            
)

parser = StrOutputParser()

extraction_chain = extraction_prompt | llm | parser

matching_chain = matching_prompt | llm | parser

scoring_chain = scoring_prompt | llm | parser

explanation_chain = explanation_prompt | llm | parser

print("✅ All 4 LCEL chains built using pipe operator (|).")
print("   extraction_chain  = extraction_prompt  | llm | parser")
print("   matching_chain    = matching_prompt    | llm | parser")
print("   scoring_chain     = scoring_prompt     | llm | parser")
print("   explanation_chain = explanation_prompt | llm | parser")

In [ ]:
from langsmith import traceable

def safe_parse_json(raw: str) -> dict:
    """Safely parse JSON from LLM output, stripping accidental fences."""
    cleaned = raw.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw_output": cleaned}


@traceable(name="resume_screening_pipeline")   
def run_screening_pipeline(resume: str, job_description: str, candidate_name: str) -> dict:
    """
    Full Resume Screening Pipeline:
      Step 1 → Skill Extraction
      Step 2 → Matching
      Step 3 → Scoring
      Step 4 → Explanation
    All steps traced via LangSmith.
    """
    print(f"\n{'='*55}")
    print(f"  Processing: {candidate_name}")
    print(f"{'='*55}")

    print("  ⏳ Step 1: Extracting skills...")
    raw_extraction = extraction_chain.invoke({"resume": resume})
    extracted      = safe_parse_json(raw_extraction)
    print(f"  ✅ Extracted profile for {extracted.get('name', candidate_name)}")

    print("  ⏳ Step 2: Matching against JD...")
    raw_matching = matching_chain.invoke({
        "extracted_profile": json.dumps(extracted),
        "job_description":   job_description
    })
    match_result = safe_parse_json(raw_matching)
    print(f"  ✅ Matched: {len(match_result.get('matched_skills', []))} skills | "
          f"Missing: {len(match_result.get('missing_skills', []))} skills")

    print("  ⏳ Step 3: Calculating score...")
    raw_score  = scoring_chain.invoke({
        "match_analysis":  json.dumps(match_result),
        "job_description": job_description
    })
    score_data = safe_parse_json(raw_score)
    print(f"  ✅ Score: {score_data.get('score', 'N/A')} / 100")

    print("  ⏳ Step 4: Generating explanation...")
    raw_explanation = explanation_chain.invoke({
        "extracted_profile": json.dumps(extracted),
        "match_analysis":    json.dumps(match_result),
        "score_breakdown":   json.dumps(score_data),
        "job_description":   job_description
    })
    explanation = safe_parse_json(raw_explanation)
    print(f"  ✅ Recommendation: {explanation.get('recommendation', 'N/A')}")

    return {
        "candidate_name":   candidate_name,
        "extracted_profile": extracted,
        "match_analysis":    match_result,
        "score_breakdown":   score_data,
        "explanation":       explanation,
    }


print("✅ Pipeline function defined with @traceable decorator.")

In [ ]:
all_results = []

for candidate in candidates:
    result = run_screening_pipeline(
        resume          = candidate["resume"],
        job_description = JOB_DESCRIPTION,
        candidate_name  = candidate["name"]
    )
    result["candidate_type"] = candidate["type"]
    all_results.append(result)

print("\n\n🏁 All 3 candidates processed. Check LangSmith for traces!")

In [ ]:
def display_result(result: dict):
    """Pretty-print a single candidate's screening result."""
    exp   = result.get("explanation", {})
    score = result.get("score_breakdown", {})
    match = result.get("match_analysis", {})
    prof  = result.get("extracted_profile", {})

    rec = exp.get("recommendation", "N/A")
    rec_icon = {"SHORTLIST": "✅", "MAYBE": "🟡", "REJECT": "❌"}.get(rec, "❓")

    print(f"\n{'─'*58}")
    print(f" 👤  {result['candidate_name']}  [{result['candidate_type']} Candidate]")
    print(f"{'─'*58}")
    print(f" 🎓  Education     : {prof.get('education', 'N/A')}")
    print(f" 📅  Experience    : {prof.get('years_experience', 0)} years")
    print(f" 🛠️   Skills found  : {', '.join(prof.get('skills', [])) or 'None'}")
    print()
    print(f" 🟢  Matched Skills: {', '.join(match.get('matched_skills', [])) or 'None'}")
    print(f" 🔴  Missing Skills: {', '.join(match.get('missing_skills', [])) or 'None'}")
    print(f" 📋  Exp Match     : {match.get('experience_match', False)}")
    print(f" 🎓  Edu Match     : {match.get('education_match', False)}")
    print()
    print(f" 📈  SCORE BREAKDOWN")
    print(f"      Skills     : {score.get('skills_points',     0)}/40")
    print(f"      Experience : {score.get('experience_points', 0)}/30")
    print(f"      Education  : {score.get('education_points',  0)}/20")
    print(f"      Bonus      : {score.get('bonus_points',       0)}/10")
    print(f"      ─────────────────")
    print(f"      TOTAL      : {score.get('score', 'N/A')}/100")
    print()
    print(f" 💬  Strengths: {exp.get('strengths', 'N/A')}")
    print(f" ⚠️   Gaps     : {exp.get('gaps', 'N/A')}")
    print(f" 📝  Summary  : {exp.get('explanation', 'N/A')}")
    print(f" {rec_icon}  RECOMMENDATION: {rec}")
    print(f"{'─'*58}")

print("\n" + "═"*58)
print("         AI RESUME SCREENING — FINAL RESULTS")
print("═"*58)

for res in all_results:
    display_result(res)

In [ ]:
import pandas as pd

rows = []
for r in all_results:
    score = r.get("score_breakdown", {})
    exp   = r.get("explanation", {})
    match = r.get("match_analysis", {})
    rows.append({
        "Candidate"     : r["candidate_name"],
        "Type"          : r["candidate_type"],
        "Score (/100)"  : score.get("score", "N/A"),
        "Matched Skills": len(match.get("matched_skills", [])),
        "Missing Skills": len(match.get("missing_skills", [])),
        "Recommendation": exp.get("recommendation", "N/A"),
    })

df = pd.DataFrame(rows)
print("\n📊 SUMMARY TABLE")
print(df.to_string(index=False))

In [ ]:
from langchain.prompts import PromptTemplate
from langsmith import traceable

bad_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Score this resume for a Data Scientist job. Resume: {resume}
"""
)

bad_chain = bad_prompt | llm | parser

@traceable(name="debug_bad_prompt_demo", tags=["debug", "incorrect_output"])
def run_bad_prompt_demo():
    """Intentionally incorrect prompt for LangSmith debugging demonstration."""
    print("\n⚠️  Running bad prompt (no format constraints)...")
    raw = bad_chain.invoke({"resume": RESUME_WEAK})
    print("📤 Raw output (unstructured, hard to parse):")
    print("-" * 50)
    print(raw)
    print("-" * 50)
    print("\n❗ Issues with this output:")
    print("   1. No JSON structure — cannot be parsed programmatically")
    print("   2. No score breakdown — just a free-text guess")
    print("   3. May hallucinate skills not in the resume")
    print("   4. No recommendation label")
    print("\n✅ Fix: Use structured PromptTemplates with explicit JSON output constraints")
    return raw

debug_output = run_bad_prompt_demo()
print("\n📌 This run is tagged 'debug' in LangSmith — find it in your traces dashboard.")

In [ ]:

output_file = "screening_results.json"

with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"✅ Full results saved to: {output_file}")
print("   Include this file in your GitHub repository.")
print()
print("🚀 Assignment Complete! Checklist:")
print("   ✅ 3 resumes processed (Strong, Average, Weak)")
print("   ✅ 4-step pipeline: Extract → Match → Score → Explain")
print("   ✅ LangChain LCEL chains with .invoke()")
print("   ✅ PromptTemplates with structured JSON output")
print("   ✅ Few-shot prompting (explanation chain)")
print("   ✅ LangSmith tracing with LANGCHAIN_TRACING_V2=true")
print("   ✅ Debug trace with intentionally bad output")
print("   ✅ Modular structure: prompts / chains / main")